In [ ]:
from helper_chat import chat,add_user_message
from helper_prompt_evaluator import PromptEvaluator

DATASET_FILENAME = 'dataset_prompting_excercise.json'

In [ ]:
# Create an instance of PromptEvaluator
# Increase `max_concurrent_tasks` for greater concurrency, but beware of rate limit errors!
evaluator = PromptEvaluator(max_concurrent_tasks=1)

In [ ]:
dataset = evaluator.generate_dataset(
    # Describe the purpose or goal of the prompt you're trying to test
    task_description=""" 
    Extract topics out of a passage of text from scholarly article into JSON array of strings
    """,
    # Describe the different inputs that your prompt requires
    prompt_inputs_spec={
        "content": "One paragraph of text from scholarly article into a json array of strings"
    },
    # Where to write the generated dataset
    output_file=DATASET_FILENAME,
    # Number of test cases to generate (recommend keeping this low if you're getting rate limit errors)
    num_cases=4,
    # force_regenerate=True,  # uncomment to overwrite existing dataset
)

In [ ]:
# define and run the prompt you want to evaluate, returning the raw model output
# This function is executed once for each test case

def run_prompt(prompt_input):
    prompt = f"""
    Extract key topics mentioned in the passage of text from scholarly journal into json array of strings.
    
    <text>
    {prompt_input["content"]}
    </text>

    Follow these steps:
    1. Closely examine the provided text.
    2. Identify each topic mentioned.
    3. Add each topic to JSON array
    4. Respond with the JSON array. Do not provide any other text or commentary.
    """
    messages = []
    add_user_message(messages,prompt)
    return chat(messages)

In [ ]:
results = evaluator.run_evaluation (
    run_prompt_function=run_prompt, 
    dataset_file=DATASET_FILENAME, 
    extra_criteria=""" 
    The output should include 
    - Contains a JSON array of strings, containing each topic mentioned in the article
    - The Strings should contain only the topics without extra commentary
    - Response should contain json array and nothing else.
    """,
    json_output_file="output_prompting_excercise.json",
    html_output_file="output_prompting_excercise.html"
)